# The `__set_name__` Method: Advanced Tutorial Problems, Volume II

**A new, step-by-step problem set with complete solutions.** This notebook follows the *teaching rhythm* of the supplied lesson: establish a small observation, ask what happens, run a short experiment, explain its output, improve the implementation, and test the improvement. It does **not** reuse the earlier problem set's implementations.

**Scope:** 15 independent but progressively connected problems, with realistic use cases, intermediate experiments, edge cases, complete reference solutions, and executable checks. Python **3.10+**, standard library only. Run cells in order from a fresh kernel.

## How to use this notebook

Every problem proceeds through **Problem → Predict → Experiment → Explain → Build → Verify → Reflect**. Try answering each prediction before running its code. The worked solutions are intentionally divided into small cells: the *reasoning* belongs in Markdown, and the executable statements belong in Python cells.

**Important correction to the source lesson's shorthand:** `__set_name__` does not run when you instantiate a descriptor. During creation of a class, `type.__new__` invokes it on attributes that provide it. A descriptor can receive multiple calls if bound to multiple names or bound during creation of multiple classes. Ordinary later assignments of an attribute to a class do not invoke it automatically. The focus here is what these facts mean for design.

### Shared test helper

We'll make failures explicit rather than relying on printed exceptions alone. `expect_error` ensures an operation really fails with the stated exception type and (optionally) the expected message. A missing-value sentinel helps us avoid confusing an unset attribute with an attribute explicitly assigned `None`.

In [1]:
import re
from decimal import Decimal
from enum import Enum
from functools import update_wrapper
from types import MethodType

MISSING = object()

def expect_error(exception_type, action, message=None):
    try:
        action()
    except exception_type as exc:
        if message is not None:
            assert re.search(message, str(exc)), (str(exc), message)
        return exc
    except Exception as exc:
        raise AssertionError(f"Expected {exception_type.__name__}, got {type(exc).__name__}") from exc
    raise AssertionError(f"Expected {exception_type.__name__}, but nothing was raised")

print("Test helper is ready.")

Test helper is ready.


### Two rules to keep in view

1. `__set_name__(self, owner, name)` identifies the class **where a descriptor was bound during class creation**, and the binding name. It does not tell `__get__` which alias caused a future lookup.
2. A descriptor with `__set__` or `__delete__` is a **data descriptor**. Such descriptors take precedence over same-name entries in an ordinary instance's `__dict__`, as long as normal object attribute lookup is used. Direct dictionary edits or custom `__getattribute__`/`__setattr__` implementations can bypass guarantees.

The notebook's examples use explicitly scoped attribute names and separate class definitions to make reruns straightforward.

---
## Problem 01 — Distinguish *missing* from `None` in a name-aware descriptor

**Scenario.** An identity field must contain a nonblank string. Access before assignment should raise `AttributeError` so introspection behaves normally. A beginner's implementation uses `dict.get(name)`, which makes an unassigned field look just like a field whose value is `None`.

**Your tasks:** demonstrate the ambiguity; write a solution using `__set_name__` and a sentinel; prove that a rejected assignment leaves the old value intact.

### Step 1 — Reproduce the ambiguous behavior

**Predict:** What two very different states both result in `None` below? Notice that we have not validated writes yet. The problem is the *getter's information loss*.

In [2]:
class NaiveIdentity01:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

class NaiveUser01:
    identity = NaiveIdentity01()

naive01 = NaiveUser01()
print("Never assigned:", naive01.identity)
naive01.__dict__["identity"] = None
print("Explicitly stored None:", naive01.identity)
assert naive01.identity is None

Never assigned: None
Explicitly stored None: None


### Step 2 — Explain the problem and specify the contract

`dict.get` returns `None` both for an absent key and for a key explicitly holding `None`. Instead, use a unique sentinel with `dict.get(name, MISSING)`, or test membership explicitly. Our final descriptor is a **data descriptor** because it defines `__set__`, and it stores values under the *same public name* in `instance.__dict__`. Normal lookup still invokes its getter.

For this exercise, an empty or whitespace-only string is invalid, leading/trailing whitespace is stripped, and only actual `str` instances are accepted.

In [3]:
class IdentityField01:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self.name, MISSING)
        if value is MISSING:
            raise AttributeError(f"{self.name} has not been assigned")
        return value

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        normalized = value.strip()
        if not normalized:
            raise ValueError(f"{self.name} must not be blank")
        instance.__dict__[self.name] = normalized

class User01:
    identity = IdentityField01()

user01 = User01()
print("Binding:", User01.identity.name)
expect_error(AttributeError, lambda: user01.identity, "identity")

Binding: identity


AttributeError('identity has not been assigned')

### Step 3 — Verify normal behavior *and* failure behavior

Assign a valid value first. Then try two invalid replacements. The tests below make sure validation happens **before** the stored value is modified.

In [4]:
user01.identity = "  Ada-42  "
assert user01.identity == "Ada-42"
assert user01.__dict__ == {"identity": "Ada-42"}
expect_error(TypeError, lambda: setattr(user01, "identity", None), "identity")
expect_error(ValueError, lambda: setattr(user01, "identity", " 	 "), "identity")
assert user01.identity == "Ada-42"
assert User01.identity is User01.__dict__["identity"]
print("Problem 01 passed: missing, normalization, and state preservation.")

Problem 01 passed: missing, normalization, and state preservation.


**Takeaway.** `__set_name__` supplies accurate diagnostic names, but the getter must separately define what *uninitialized* means. An `AttributeError` for missing data is often more useful than silently returning `None`.

---
## Problem 02 — Why assigning one descriptor to two names is ambiguous

**Scenario.** A descriptor author assumes `__set_name__` is always called exactly once. A teammate writes `left = right = descriptor` in a class body. We will see two callbacks and learn why saving only `self.name` is not a general solution for aliases.

**Challenge:** identify exactly what information `__get__` does, and does not, receive on later attribute access; then write an unambiguous version.

### Step 1 — Observe class creation

Make a small descriptor that records *each* callback. Before running, predict the length and contents of `bindings02` and the value left in `shared02.name`.

In [5]:
bindings02 = []

class AliasProbe02:
    def __set_name__(self, owner, name):
        bindings02.append((owner.__name__, name))
        self.name = name

    def __get__(self, instance, owner):
        return self if instance is None else f"configured as {self.name}"

shared02 = AliasProbe02()

class Ambiguous02:
    left = shared02
    right = shared02

print("Bindings:", bindings02)
print("Lookups:", Ambiguous02().left, Ambiguous02().right)
assert bindings02 == [("Ambiguous02", "left"), ("Ambiguous02", "right")]
assert Ambiguous02().left == Ambiguous02().right == "configured as right"

Bindings: [('Ambiguous02', 'left'), ('Ambiguous02', 'right')]
Lookups: configured as right configured as right


### Step 2 — Explain the information gap

`__get__(instance, owner)` receives an instance and a class; there is **no argument for the name used at the lookup site**. Both class attributes point at the same descriptor object, so overwriting a single `self.name` loses the first name. Keeping a list of aliases records history but still cannot tell which alias was used for a particular instance lookup.

For independent fields, construct independent descriptor objects. An intentional alias should instead be designed as an alias (for example, a `property` forwarding to a canonical field).

In [6]:
class NamedValue02:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class Unambiguous02:
    left = NamedValue02()
    right = NamedValue02()

    @property
    def left_alias(self):
        return self.left

    @left_alias.setter
    def left_alias(self, value):
        self.left = value

sample02 = Unambiguous02()
sample02.left, sample02.right = 10, 20
sample02.left_alias = 30
assert (sample02.left, sample02.right, sample02.left_alias) == (30, 20, 30)
assert Unambiguous02.left is not Unambiguous02.right
print("Problem 02 passed: independent state plus an explicit alias.")

Problem 02 passed: independent state plus an explicit alias.


**Takeaway.** `__set_name__` identifies each class-body binding; it does not magically make one stateful descriptor support multiple field identities. Design aliases as a separate, explicit feature.

---
## Problem 03 — One reusable descriptor across unrelated owners

**Scenario.** Unlike the accidental aliases in Problem 02, a library intentionally reuses *one* descriptor object in **two different classes**, but under different names. A single `self.name` would cause cross-class contamination.

**Challenge:** store the binding **per defining class**, resolve it via the instance's method-resolution order (MRO), and preserve inherited access. This is a deliberate multi-owner design; it supports one binding per owner and is not intended for two aliases within a single class.

### Step 1 — Think through inheritance

If `Child` inherits `Base.field`, Python does not create a new binding in `Child` and therefore does not call `__set_name__` again. The callback's `owner` is the **defining owner**; the `owner` later supplied to `__get__` can be a subclass. We therefore need to look through the MRO to find the class that registered this descriptor.

In [7]:
class ReusableField03:
    def __init__(self):
        self.names_by_owner = {}

    def __set_name__(self, owner, name):
        if owner in self.names_by_owner and self.names_by_owner[owner] != name:
            raise ValueError("Only one binding per owner is supported")
        self.names_by_owner[owner] = name

    def _name_for(self, cls):
        for candidate in cls.__mro__:
            if candidate in self.names_by_owner:
                return self.names_by_owner[candidate]
        raise AttributeError("Descriptor is not bound in this MRO")

    def __get__(self, instance, owner):
        if instance is None:
            return self
        name = self._name_for(type(instance))
        return instance.__dict__[name]

    def __set__(self, instance, value):
        name = self._name_for(type(instance))
        instance.__dict__[name] = value

shared03 = ReusableField03()

class Invoice03:
    reference = shared03

class Shipment03:
    tracking_code = shared03

class PriorityShipment03(Shipment03):
    pass

print({cls.__name__: name for cls, name in shared03.names_by_owner.items()})

{'Invoice03': 'reference', 'Shipment03': 'tracking_code'}


### Step 2 — Test all three access paths

For unrelated classes, the first class in the instance's MRO with a registered binding supplies the name. For a subclass, the inherited defining class supplies the existing name. There is no mutable global “current name” to overwrite.

In [8]:
invoice03 = Invoice03()
shipment03 = Shipment03()
priority03 = PriorityShipment03()
invoice03.reference = "INV-8"
shipment03.tracking_code = "TRK-3"
priority03.tracking_code = "TRK-9"
assert invoice03.__dict__ == {"reference": "INV-8"}
assert shipment03.__dict__ == {"tracking_code": "TRK-3"}
assert priority03.tracking_code == "TRK-9"
assert len(shared03.names_by_owner) == 2  # No new callback on inheritance.
print("Problem 03 passed: two defining owners and inherited lookup.")

Problem 03 passed: two defining owners and inherited lookup.


### Step 3 — Recognize the remaining restriction

This design is safe for its stated contract, but it is **not** an alias solution. If two names in the *same* class refer to this object, the second `__set_name__` call rejects the class definition. In recent Python versions, errors raised by `__set_name__` may propagate directly; in some older versions, class creation wraps them in `RuntimeError`. We handle that diagnostic explicitly in Problem 14.

---
## Problem 04 — Lazy defaults without sharing mutable values

**Scenario.** Every new task should have a fresh list of labels, created only on first access. Writing `default=[]` inside a descriptor risks one list shared across unrelated instances.

**Challenge:** use `__set_name__` for per-instance storage, a `default_factory` to create fresh lists, and a validation step before replacement.

### Step 1 — Demonstrate the accidental sharing

The naive descriptor keeps the list on the descriptor object, not on the model instance. Since the descriptor lives on the **class**, both instances see the same list.

In [9]:
class SharedTags04:
    def __init__(self):
        self.value = []

    def __get__(self, instance, owner):
        return self if instance is None else self.value

class BadTask04:
    tags = SharedTags04()

first_bad04, second_bad04 = BadTask04(), BadTask04()
first_bad04.tags.append("urgent")
assert second_bad04.tags == ["urgent"]
print("Shared state (bug):", first_bad04.tags is second_bad04.tags)

Shared state (bug): True


### Step 2 — Write the corrected descriptor

The factory is called only when the instance lacks a value. For this demonstration, we validate *whole-list replacements* and make a defensive copy. Directly mutating a list returned by the getter is still possible; descriptor `__set__` cannot intercept `obj.tags.append(...)`. That limitation is part of the problem, not something to hide.

In [10]:
class LazyTags04:
    def __init__(self, default_factory=list):
        self.default_factory = default_factory

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            instance.__dict__[self.name] = self.default_factory()
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, list) or any(not isinstance(x, str) for x in value):
            raise TypeError(f"{self.name} must be a list of strings")
        instance.__dict__[self.name] = list(value)

class Task04:
    tags = LazyTags04()

first04, second04 = Task04(), Task04()
assert first04.__dict__ == {}  # Lazy allocation.
first04.tags.append("urgent")
assert second04.tags == []
assert first04.tags is not second04.tags
print("Separated instance state:", first04.__dict__, second04.__dict__)

Separated instance state: {'tags': ['urgent']} {'tags': []}


### Step 3 — Validate replacement and identify the boundary

Notice that failed replacement does not discard the original list. But an in-place append reaches the *list object*, not `Task04.__setattr__`. A stricter API would return an immutable tuple or a custom validated collection.

In [11]:
first04.tags = ["open", "assigned"]
external04 = ["draft"]
second04.tags = external04
external04.append("changed later")
assert second04.tags == ["draft"]
expect_error(TypeError, lambda: setattr(first04, "tags", ["ok", 42]), "tags")
assert first04.tags == ["open", "assigned"]
print("Problem 04 passed: lazy isolation, defensive copy, invalid-write safety.")

Problem 04 passed: lazy isolation, defensive copy, invalid-write safety.


**Takeaway.** `__set_name__` gives us a reliable per-field key. A factory gives us per-instance mutable data. Assignment validation and mutation validation are **different** contracts.

---
## Problem 05 — Cross-field range validation without partial writes

**Scenario.** A range model exposes `minimum` and `maximum`. Once both are assigned, `minimum <= maximum` must hold. Either field can be initialized first. A rejected update must preserve the previous valid range.

**Challenge:** let each descriptor know the name of its partner; validate the partner's declaration during `__set_name__`; validate pair consistency before writing any value.

### Step 1 — Separate declaration-time checks from runtime checks

At class creation, the new class namespace already contains *both* declarations. Our `__set_name__` can check `vars(owner)[partner]` without triggering descriptor access. But an instance's actual numeric values do not exist yet, so comparisons belong in `__set__`.

In [12]:
class RangeEndpoint05:
    def __init__(self, partner, *, is_minimum):
        self.partner = partner
        self.is_minimum = is_minimum

    def __set_name__(self, owner, name):
        self.name = name
        candidate = vars(owner).get(self.partner)
        if not isinstance(candidate, RangeEndpoint05):
            raise TypeError(f"{name}: expected descriptor partner {self.partner}")
        if candidate.is_minimum == self.is_minimum:
            raise TypeError(f"{name}: endpoints must have opposite roles")

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} is unset")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if type(value) not in (int, float):
            raise TypeError(f"{self.name} must be an int or float, excluding bool")
        other = instance.__dict__.get(self.partner, MISSING)
        if other is not MISSING:
            valid = value <= other if self.is_minimum else other <= value
            if not valid:
                raise ValueError(f"{self.name} conflicts with {self.partner}")
        instance.__dict__[self.name] = value

class Interval05:
    minimum = RangeEndpoint05("maximum", is_minimum=True)
    maximum = RangeEndpoint05("minimum", is_minimum=False)

### Step 2 — Try both initialization orders

Because comparison is conditional on the *other value already being present*, either order works for a valid final interval. This validates individual assignments; it does not attempt to implement multi-field transactions.

In [13]:
left05, right05 = Interval05(), Interval05()
left05.minimum, left05.maximum = 1, 10
right05.maximum, right05.minimum = 10, 1
assert (left05.minimum, left05.maximum) == (right05.minimum, right05.maximum)
expect_error(ValueError, lambda: setattr(left05, "minimum", 11), "minimum")
expect_error(ValueError, lambda: setattr(right05, "maximum", -1), "maximum")
expect_error(TypeError, lambda: setattr(left05, "minimum", True), "minimum")
assert (left05.minimum, left05.maximum) == (1, 10)
assert (right05.minimum, right05.maximum) == (1, 10)
print("Problem 05 passed: both orders, cross-field checks, preserved state.")

Problem 05 passed: both orders, cross-field checks, preserved state.


### Step 3 — Analyze the boundary

This pattern is practical for small models whose fields are assigned separately. It does not guarantee global atomicity for *multiple* assignments; if you need to move `[1, 10]` to `[20, 30]`, a coordinated update method should validate the proposed entire state first. Also, floating-point NaN makes ordinary ordering checks tricky; a production range validator should reject non-finite floats explicitly.

---
## Problem 06 — Declarative query columns from `__set_name__`

**Scenario.** We want a tiny, in-memory query builder: writing `Customer06.email == "someone@example.test"` should make a parameterized equality predicate, not run a database query. Accessing the *same descriptor on an instance* should return a stored and validated value.

**Challenge:** derive the column name during class creation, distinguish class and instance access, and keep user-supplied values separate from SQL strings. This is an educational expression builder, **not** a complete SQL engine.

### Step 1 — Design the two access modes

`Customer06.email` calls `Column06.__get__(None, Customer06)` and returns the descriptor itself, letting us construct an expression. `customer.email` calls `__get__` with a real instance and returns that instance's value. The binding name comes from `__set_name__`; the example's table name comes from the defining class name.

We overload `==` only for building predicates. As a result, equality on the column object is *not* ordinary Boolean equality; this is deliberate for a query-expression DSL.

In [14]:
class Predicate06:
    def __init__(self, sql, params):
        self.sql = sql
        self.params = tuple(params)

class Column06:
    def __set_name__(self, owner, name):
        self.table = owner.__name__.lower()
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} is unset")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{self.name} must be nonblank text")
        instance.__dict__[self.name] = value

    def __eq__(self, other):
        return Predicate06(f"{self.table}.{self.name} = %s", (other,))

class Customer06:
    email = Column06()
    region = Column06()

predicate06 = Customer06.email == "a@example.test"
print("SQL fragment:", predicate06.sql)
print("Parameters:", predicate06.params)

SQL fragment: customer06.email = %s
Parameters: ('a@example.test',)


### Step 2 — Build the simplest complete query

Identifiers are generated from trusted *class declarations* for this exercise, while the value is placed in `params`, not interpolated into SQL. DB-API placeholder styles vary by database; `%s` is chosen only as a readable example here. A real implementation must also quote/validate identifiers using its database driver's rules.

In [15]:
class Query06:
    def __init__(self, model):
        self.model = model

    def where(self, predicate):
        if not isinstance(predicate, Predicate06):
            raise TypeError("where expects a Predicate06")
        table = self.model.__name__.lower()
        # Avoid composing predicates from columns of an unrelated table.
        prefix = table + "."
        if not predicate.sql.startswith(prefix):
            raise ValueError("Predicate belongs to another model")
        return f"SELECT * FROM {table} WHERE {predicate.sql}", predicate.params

sql06, params06 = Query06(Customer06).where(
    Customer06.email == "x' OR 1=1 --"
)
print(sql06)
print(params06)
assert sql06 == "SELECT * FROM customer06 WHERE customer06.email = %s"
assert params06 == ("x' OR 1=1 --",)

SELECT * FROM customer06 WHERE customer06.email = %s
("x' OR 1=1 --",)


### Step 3 — Verify ordinary instance access remains independent

The same class attribute serves two roles. We also check rejection of invalid values and confirm that `__set_name__` captured *each* declared column name independently.

In [16]:
customer06 = Customer06()
customer06.email = "ada@example.test"
customer06.region = "EU"
assert (customer06.email, customer06.region) == ("ada@example.test", "EU")
assert Customer06.email.name == "email"
assert Customer06.region.name == "region"
expect_error(ValueError, lambda: setattr(customer06, "email", "  "), "email")
assert customer06.email == "ada@example.test"
print("Problem 06 passed: class-level expressions and instance-level values.")

Problem 06 passed: class-level expressions and instance-level values.


**Takeaway.** Class-level descriptor access is valuable for declarative APIs. `__set_name__` can populate field metadata once, while `__get__` selects behavior based on whether `instance is None`.

---
## Problem 07 — A custom `__getattribute__` can bypass the descriptor protocol

**Scenario.** A logging mixin intercepts *every* attribute read. Its first implementation checks `self.__dict__` before delegating. Suddenly, a field descriptor that ought to transform the value is silently ignored.

**Challenge:** reproduce the bug; repair the override without recursive attribute lookup; show how the fix preserves data-descriptor precedence.

### Step 1 — Set up a visibly transforming getter

Our field stores a raw number but returns it multiplied by ten. With normal `object.__getattribute__`, the descriptor wins over the same-name entry in the instance dictionary. Try to predict the output of the broken class.

In [17]:
class ScaleField07:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name] * 10

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class BrokenRead07:
    scale = ScaleField07()

    def __getattribute__(self, name):
        data = object.__getattribute__(self, "__dict__")
        if name in data:
            return data[name]  # Incorrect: bypasses data descriptor.
        return object.__getattribute__(self, name)

broken07 = BrokenRead07()
broken07.scale = 7
print("Bypassed result:", broken07.scale)
assert broken07.scale == 7  # Wrong for the field's public contract.

Bypassed result: 7


### Step 2 — Fix the delegation point

The safe default is to let `object.__getattribute__` perform the full lookup first. An override can log the result and return it. We log only public reads and skip internal `__dict__` accesses. We read the log via `object.__getattribute__(self, "_reads")` to avoid accidental recursion.

In [18]:
class LoggedRead07:
    scale = ScaleField07()

    def __init__(self):
        self._reads = []

    def __getattribute__(self, name):
        result = object.__getattribute__(self, name)
        if name not in ("_reads", "__dict__"):
            log = object.__getattribute__(self, "_reads")
            log.append(name)
        return result

logged07 = LoggedRead07()
logged07.scale = 7
assert logged07.scale == 70
assert logged07.__dict__["scale"] == 7
assert logged07._reads == ["scale"]
print("Descriptor result:", logged07.scale)

Descriptor result: 70


### Step 3 — Check a missing field, too

By delegating to `object.__getattribute__`, we preserve the usual error behavior for absent names. A custom hook may deliberately change semantics, but it should do so consciously, not accidentally by reading the instance dictionary first.

In [19]:
expect_error(AttributeError, lambda: logged07.nonexistent)
assert logged07.scale == 70
print("Problem 07 passed: logging and descriptor semantics coexist.")

Problem 07 passed: logging and descriptor semantics coexist.


**Takeaway.** Descriptor precedence describes Python's *normal* lookup algorithm. If you replace that algorithm in `__getattribute__`, you become responsible for reproducing or delegating its behavior.

---
## Problem 08 — A custom `__setattr__` can bypass validation

**Scenario.** A model overrides `__setattr__` to keep a change log. One implementation stores directly in `__dict__`. The ordinary assignment `obj.quantity = -10` then bypasses the descriptor's `__set__` validator.

**Challenge:** build a positive-integer field, compare the broken writer with a delegated writer, and ensure the log records only successful assignments.

### Step 1 — Create a field whose `__set__` must run

A strict integer check rejects `bool`, which is a subclass of `int`. We write the value only *after* validating it.

In [20]:
class PositiveInt08:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if type(value) is not int or value <= 0:
            raise ValueError(f"{self.name} must be a positive int")
        instance.__dict__[self.name] = value

class BrokenWrite08:
    quantity = PositiveInt08()

    def __setattr__(self, name, value):
        self.__dict__[name] = value  # Incorrect for descriptor-backed fields.

broken08 = BrokenWrite08()
broken08.quantity = -10
assert broken08.quantity == -10
print("Invalid quantity accepted by broken override:", broken08.quantity)

Invalid quantity accepted by broken override: -10


### Step 2 — Delegate first; audit after success

`object.__setattr__(self, name, value)` performs normal assignment and invokes data-descriptor `__set__` where applicable. For a reliable successful-write log, append the event only if delegation returned without raising. Retrieve the log with `object.__getattribute__` so this hook does not accidentally route through another user-defined accessor.

In [21]:
class AuditedWrite08:
    quantity = PositiveInt08()

    def __init__(self):
        object.__setattr__(self, "_writes", [])

    def __setattr__(self, name, value):
        object.__setattr__(self, name, value)
        if name != "_writes":
            object.__getattribute__(self, "_writes").append((name, value))

good08 = AuditedWrite08()
good08.quantity = 5
expect_error(ValueError, lambda: setattr(good08, "quantity", -10), "quantity")
expect_error(ValueError, lambda: setattr(good08, "quantity", True), "quantity")
assert good08.quantity == 5
assert good08._writes == [("quantity", 5)]
print("Accepted writes:", good08._writes)

Accepted writes: [('quantity', 5)]


### Step 3 — Name the remaining escape hatch

Users can still mutate `good08.__dict__["quantity"]` directly. Python descriptors implement an attribute-access protocol, **not** an unbreakable integrity or security boundary. For a controlled internal model, document that direct storage mutation is unsupported.

In [22]:
assert AuditedWrite08.quantity.name == "quantity"
assert good08.__dict__["quantity"] == 5
print("Problem 08 passed: validation runs before the audit event.")

Problem 08 passed: validation runs before the audit event.


---
## Problem 09 — Which class arrives as `owner` on inherited and `super()` access?

**Scenario.** A descriptor captures the class that originally defines it in `__set_name__`. Later, a subclass inherits that field. Someone assumes the `owner` passed to every `__get__` is always the original class.

**Challenge:** capture all three access paths: class-level, instance-level, and `super()` lookup. Explain why *binding owner* and *access owner* must not be conflated.

### Step 1 — Record the defining owner exactly once

We use a tuple of simple names instead of printing object addresses, so notebook output is deterministic. The getter returns the descriptor for class access and a useful payload for instance access.

In [23]:
class OwnerProbe09:
    def __init__(self):
        self.binding_events = []
        self.access_events = []

    def __set_name__(self, owner, name):
        self.binding_events.append((owner.__name__, name))

    def __get__(self, instance, owner):
        self.access_events.append((instance is None, owner.__name__))
        return self if instance is None else owner.__name__

class Base09:
    marker = OwnerProbe09()

class Child09(Base09):
    pass

child09 = Child09()
assert Child09.marker is Base09.marker
print("Binding:", Base09.marker.binding_events)

Binding: [('Base09', 'marker')]


### Step 2 — Compare the access events

When accessing an inherited descriptor through the subclass, Python normally supplies the subclass as `owner`. Using `super(Child09, child09)` changes *where Python starts its attribute search*, but does not change the concrete instance's type supplied as the access owner here.

In [24]:
probe09 = Base09.__dict__["marker"]
probe09.access_events.clear()
class_result09 = Child09.marker
ordinary_result09 = child09.marker
super_result09 = super(Child09, child09).marker
print("Access events:", probe09.access_events)
assert class_result09 is probe09
assert ordinary_result09 == "Child09"
assert super_result09 == "Child09"
assert probe09.access_events == [
    (True, "Child09"),
    (False, "Child09"),
    (False, "Child09"),
]
assert probe09.binding_events == [("Base09", "marker")]

Access events: [(True, 'Child09'), (False, 'Child09'), (False, 'Child09')]


### Step 3 — Translate observation into a design rule

Use the class from `__set_name__` when you need to know *where a field was declared*. Use `owner` received by `__get__` when you want access-context information. Note that `__set__` has **no owner argument**; if needed, derive access type from `type(instance)`. Keep these meanings separate in descriptor frameworks.

In [25]:
assert probe09.binding_events[0][0] != probe09.access_events[0][1]
print("Problem 09 passed: defining owner differs from access owner.")

Problem 09 passed: defining owner differs from access owner.


---
## Problem 10 — Build an auditable descriptor with deletion semantics

**Scenario.** A configuration field should record all successful writes and deletes, including its own field name. Reading an uninitialized or deleted field should raise `AttributeError`. Failed operations must leave both state and audit history unchanged.

**Challenge:** implement `__get__`, `__set__`, `__delete__`, and `__set_name__` together. This problem studies a *complete auditing field*, rather than a deletion-only precedence trick.

### Step 1 — Define an event contract

We'll store a per-instance audit log as a list under a private implementation key. The event format is `(field_name, operation, old_value, new_value)`, using an `Enum` sentinel to represent the unset state without colliding with any valid string value. Read operations do not alter history.

We will add events **after** successful storage changes. Unsuccessful deletion is an error rather than a silent no-op.

In [26]:
class AuditState10(Enum):
    UNSET = "unset"

class AuditedField10:
    AUDIT_KEY = "_field_events_10"

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} is unset")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, str) or not value:
            raise ValueError(f"{self.name} must be nonempty text")
        values = instance.__dict__
        before = values.get(self.name, AuditState10.UNSET)
        values[self.name] = value
        values.setdefault(self.AUDIT_KEY, []).append(
            (self.name, "set", before, value)
        )

    def __delete__(self, instance):
        values = instance.__dict__
        if self.name not in values:
            raise AttributeError(f"{self.name} is already unset")
        before = values.pop(self.name)
        values.setdefault(self.AUDIT_KEY, []).append(
            (self.name, "delete", before, AuditState10.UNSET)
        )

class Settings10:
    hostname = AuditedField10()
    environment = AuditedField10()

### Step 2 — Walk through a valid lifecycle

Predict how many events will be recorded after two assignments and one deletion. Notice that the descriptor itself is shared at class level, but the audit log lives on each *instance*.

In [27]:
settings10 = Settings10()
settings10.hostname = "db.internal"
settings10.environment = "staging"
del settings10.hostname
expect_error(AttributeError, lambda: settings10.hostname, "hostname")
expected10 = [
    ("hostname", "set", AuditState10.UNSET, "db.internal"),
    ("environment", "set", AuditState10.UNSET, "staging"),
    ("hostname", "delete", "db.internal", AuditState10.UNSET),
]
assert settings10.__dict__[AuditedField10.AUDIT_KEY] == expected10
print("Audit trail:")
for event in expected10:
    print(" ", event)

Audit trail:
  ('hostname', 'set', <AuditState10.UNSET: 'unset'>, 'db.internal')
  ('environment', 'set', <AuditState10.UNSET: 'unset'>, 'staging')
  ('hostname', 'delete', 'db.internal', <AuditState10.UNSET: 'unset'>)


### Step 3 — Ensure errors do not create events

We compare a **snapshot** of the log before and after invalid operations. That catches a subtle bug where an implementation logs an attempted change before validation, or logs a deletion that never happened.

In [28]:
before10 = list(settings10.__dict__[AuditedField10.AUDIT_KEY])
expect_error(ValueError, lambda: setattr(settings10, "environment", ""), "environment")
expect_error(AttributeError, lambda: delattr(settings10, "hostname"), "hostname")
assert settings10.__dict__[AuditedField10.AUDIT_KEY] == before10
assert settings10.environment == "staging"
other10 = Settings10()
assert AuditedField10.AUDIT_KEY not in other10.__dict__
other10.hostname = "<unset>"
assert other10.__dict__[AuditedField10.AUDIT_KEY][0][2] is AuditState10.UNSET
assert other10.__dict__[AuditedField10.AUDIT_KEY][0][3] == "<unset>"
print("Problem 10 passed: successful events only, isolated logs.")

Problem 10 passed: successful events only, isolated logs.


**Production note.** A mutable audit log stored in a public object dictionary can be tampered with. This is a transparent learning example, not a tamper-proof financial or security audit system. Choose stronger storage and transaction guarantees when those are required.

---
## Problem 11 — Cache a computed *data descriptor* and invalidate it correctly

**Scenario.** A rectangle's `area` is computed from `width` and `height`. We want to cache it after the first read and invalidate only after successful updates to either dimension. `area` must remain read-only to ordinary assignment.

**Challenge:** combine three name-aware descriptors. Unlike a non-data cached descriptor that can be shadowed, the computed field here is a **data descriptor** because it defines a rejecting `__set__`.

### Step 1 — Choose a cache strategy

The area descriptor can store its cached result at `instance.__dict__["area"]`, *the same name as the data descriptor*. Normal lookup still calls `Area11.__get__` first. Both dimension descriptors must remove that cache **after** a validated write.

If a proposed dimension is invalid, the old dimension and cached area should remain unchanged. We'll also count real computations to observe the cache.

In [29]:
import math

class Dimension11:
    def __init__(self, dependent_cache_name):
        self.dependent_cache_name = dependent_cache_name

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} is unset")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if type(value) not in (int, float) or not math.isfinite(value) or value < 0:
            raise ValueError(f"{self.name} must be a finite nonnegative number")
        instance.__dict__[self.name] = value
        instance.__dict__.pop(self.dependent_cache_name, None)

class Area11:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            result = instance.width * instance.height
            instance.__dict__[self.name] = result
            instance.computations += 1
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        raise AttributeError(f"{self.name} is computed; assign dimensions instead")

class Rectangle11:
    width = Dimension11("area")
    height = Dimension11("area")
    area = Area11()

    def __init__(self, width, height):
        self.computations = 0
        self.width = width
        self.height = height

### Step 2 — Track the cache lifecycle

Reading `area` twice should compute once. Updating `width` should invalidate and recompute on the next read. A rejected update should preserve the existing valid cache. Finally, attempts to assign the computed value directly should fail.

In [30]:
rectangle11 = Rectangle11(3, 4)
assert "area" not in rectangle11.__dict__
assert rectangle11.area == 12
assert rectangle11.area == 12
assert rectangle11.computations == 1
rectangle11.width = 5
assert "area" not in rectangle11.__dict__
assert rectangle11.area == 20
assert rectangle11.computations == 2
expect_error(ValueError, lambda: setattr(rectangle11, "height", -4), "height")
assert rectangle11.area == 20
assert rectangle11.computations == 2
expect_error(AttributeError, lambda: setattr(rectangle11, "area", 999), "area")
assert rectangle11.area == 20
print("Area and computations:", rectangle11.area, rectangle11.computations)

Area and computations: 20 2


### Step 3 — Test a less obvious invalidation path

A *successful* assignment of the same numerical value still invalidates in this simple design. Optimizing away unnecessary invalidation is possible, but that is a separate design choice that requires correct equality behavior for all allowed value types.

In [31]:
rectangle11.height = 4
assert "area" not in rectangle11.__dict__
assert rectangle11.area == 20
assert rectangle11.computations == 3
assert Rectangle11.area.name == "area"
print("Problem 11 passed: cache, invalidation, and failed-update invariants.")

Problem 11 passed: cache, invalidation, and failed-update invariants.


**Takeaway.** The same-name dictionary entry is safe under ordinary access because `Area11` is a data descriptor. Explicit invalidation is essential: merely caching a computed value does not establish when it becomes stale.

---
## Problem 12 — Build a method-validating descriptor decorator

**Scenario.** Descriptors are not only for stored fields. Ordinary Python functions already implement descriptor binding. We will wrap a function in our own descriptor that remembers the method's name and ensures its **return value** is a nonnegative number.

**Challenge:** implement `__set_name__`, method binding in `__get__`, and a callable wrapper in `__call__`. Keep a correct public docstring and function name when possible.

### Step 1 — Understand where the method's instance comes from

On `Account12.projected_balance`, `instance is None`: return the descriptor for introspection. On `account.projected_balance`, create a bound method using `MethodType(self, instance)`. Calling it then invokes our descriptor object's `__call__(instance, ...)`, which in turn calls the wrapped function.

We use `functools.update_wrapper` to copy conventional metadata; this is still a simplified decorator and not a full signature-preservation library.

In [32]:
class NonnegativeResult12:
    def __init__(self, function):
        self.function = function
        update_wrapper(self, function)

    def __set_name__(self, owner, name):
        self.name = name
        self.defining_owner = owner

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def __call__(self, instance, *args, **kwargs):
        result = self.function(instance, *args, **kwargs)
        if type(result) not in (int, float) or result < 0 or not math.isfinite(result):
            raise ValueError(f"{self.name} must return a finite nonnegative number")
        return result

class Account12:
    def __init__(self, balance):
        self.balance = balance

    @NonnegativeResult12
    def projected_balance(self, withdrawal):
        """Compute the balance after a hypothetical withdrawal."""
        return self.balance - withdrawal

account12 = Account12(100)
print("Descriptor name:", Account12.projected_balance.name)
print("Bound call:", account12.projected_balance(35))

Descriptor name: projected_balance
Bound call: 65


### Step 2 — Prove binding and validation

A bound method should hold the account as `__self__`, while `__func__` refers to the descriptor wrapper. A negative projected balance must raise an error that includes the name captured by `__set_name__`.

In [33]:
bound12 = account12.projected_balance
assert isinstance(bound12, MethodType)
assert bound12.__self__ is account12
assert bound12.__func__ is Account12.projected_balance
assert bound12(35) == 65
assert Account12.projected_balance.__name__ == "projected_balance"
assert "hypothetical withdrawal" in Account12.projected_balance.__doc__
expect_error(ValueError, lambda: bound12(101), "projected_balance")
assert account12.balance == 100
print("Problem 12 passed: method binding, metadata, return validation.")

Problem 12 passed: method binding, metadata, return validation.


### Step 3 — Consider side effects

The check occurs **after** the wrapped function finishes. If that function mutates an account before returning a bad value, this wrapper cannot automatically undo the mutation. We deliberately used a pure projection method to make the contract clear. For side-effecting business operations, validate inputs *before* mutation or use an explicit transaction protocol.

---
## Problem 13 — Inherited field schemas without a metaclass

**Scenario.** A model library wants `Model.schema()` to show the names, declared types, and original owners of descriptor fields. Subclasses may add fields or override inherited ones. A base class's schema must not be mutated when a subclass is defined.

**Challenge:** use `__set_name__` to tag each field and `__init_subclass__` to construct a fresh registry after all locally declared fields have received their names. No custom metaclass is needed for this educational design.

### Step 1 — Give a field self-describing metadata

Each field captures `(owner, name)` during class creation. It also validates ordinary instance assignments. We define just two simple field kinds to keep the focus on registration rather than a large type system.

In [34]:
class Field13:
    def __init__(self, kind):
        if kind not in (str, int):
            raise ValueError("Only str and int are supported in this exercise")
        self.kind = kind

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if type(value) is not self.kind:
            raise TypeError(f"{self.name} must be {self.kind.__name__}")
        instance.__dict__[self.name] = value

### Step 2 — Collect inherited and local declarations in the right order

`__init_subclass__` runs after descriptor name setup for the new class. Build a **new dictionary**, starting with inherited schemas from the MRO. Then walk `vars(cls)` so local descriptor declarations update the registry and local *non-descriptor* definitions can intentionally hide inherited entries.

`vars(cls)` looks at the class's own namespace, which avoids accidentally treating every inherited attribute as a new declaration. Rebuilding instead of mutating a parent's dictionary prevents schema changes in a child from affecting its parent.

In [35]:
class SchemaModel13:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        merged = {}
        for base in reversed(cls.__mro__[1:]):
            merged.update(vars(base).get("_schema13", {}))
        for name, value in vars(cls).items():
            if isinstance(value, Field13):
                merged[name] = (value.kind.__name__, value.owner.__name__)
            elif name in merged:
                del merged[name]  # Explicit local non-field hides inherited field.
        cls._schema13 = merged

    @classmethod
    def schema(cls):
        return dict(cls._schema13)  # Return a defensive copy.

class Record13(SchemaModel13):
    identifier = Field13(int)
    category = Field13(str)

class CustomerRecord13(Record13):
    email = Field13(str)
    category = Field13(str)  # Override with a new descriptor.

class ArchivedRecord13(CustomerRecord13):
    category = "archived"  # No longer a descriptor field.

print("Record:", Record13.schema())
print("Customer:", CustomerRecord13.schema())
print("Archived:", ArchivedRecord13.schema())

Record: {'identifier': ('int', 'Record13'), 'category': ('str', 'Record13')}
Customer: {'identifier': ('int', 'Record13'), 'category': ('str', 'CustomerRecord13'), 'email': ('str', 'CustomerRecord13')}
Archived: {'identifier': ('int', 'Record13'), 'email': ('str', 'CustomerRecord13')}


### Step 3 — Verify inheritance, override, and isolation

A child override reports the child's declaring class, while inherited fields retain their original owners. Removing a field from a child registry should not remove it from the parent. The returned schema is a copy, so callers cannot mutate the internal dictionary by accident.

In [36]:
assert Record13.schema() == {
    "identifier": ("int", "Record13"),
    "category": ("str", "Record13"),
}
assert CustomerRecord13.schema()["identifier"] == ("int", "Record13")
assert CustomerRecord13.schema()["category"] == ("str", "CustomerRecord13")
assert CustomerRecord13.schema()["email"] == ("str", "CustomerRecord13")
assert "category" not in ArchivedRecord13.schema()
assert "category" in Record13.schema()
external13 = CustomerRecord13.schema()
external13.clear()
assert len(CustomerRecord13.schema()) == 3
customer13 = CustomerRecord13()
customer13.identifier = 7
customer13.email = "x@example.test"
expect_error(TypeError, lambda: setattr(customer13, "identifier", True), "identifier")
assert customer13.identifier == 7
print("Problem 13 passed: schema ownership, override, and independence.")

Problem 13 passed: schema ownership, override, and independence.


### Step 4 — Know when this simple approach needs more work

This registry is designed for straightforward MRO-based field discovery. Real frameworks may also need conflict detection in diamond inheritance, class decorators that add descriptors later, field ordering policies, custom metaclasses, or serialization contracts. `__set_name__` supplies *binding metadata*; it does not solve all those framework questions by itself.

---
## Problem 14 — Debug a failure inside `__set_name__` across Python versions

**Scenario.** A descriptor checks declarations during class creation. The declaration is invalid, so `__set_name__` raises `ValueError`. Depending on Python version and class-construction path, this may reach the caller directly or be wrapped in `RuntimeError` with the original error as its cause.

**Challenge:** capture the underlying failure portably, verify that the intended callback actually ran, and explain how to minimize class-creation side effects.

### Step 1 — Deliberately construct a broken class

The class statement is inside a function so the notebook can catch the exception and continue. We record one callback event before raising, then inspect the exception chain. **Do not hard-code** that the top-level exception is always `RuntimeError` or always `ValueError`.

In [37]:
events14 = []

class RequiresPrefix14:
    def __set_name__(self, owner, name):
        events14.append((owner.__name__, name))
        if not name.startswith("field_"):
            raise ValueError(f"{name} must start with field_")
        self.name = name

def define_broken14():
    class Broken14:
        broken = RequiresPrefix14()
    return Broken14

try:
    define_broken14()
except Exception as exc14:
    outer14 = exc14
    original14 = exc14.__cause__ or exc14
else:
    raise AssertionError("Class creation should have failed")

print("Observed outer exception:", type(outer14).__name__)
print("Underlying issue:", type(original14).__name__, str(original14))
assert isinstance(original14, ValueError)
assert "broken must start with field_" in str(original14)
assert events14 == [("Broken14", "broken")]

Observed outer exception: ValueError
Underlying issue: ValueError broken must start with field_


### Step 2 — Define a valid class using the same rule

A successful callback writes the field name. We check the raw class dictionary rather than invoking some unrelated getter. The intentionally failed class is never returned by `define_broken14`, so no invalid class is used by later cells.

In [38]:
class Valid14:
    field_status = RequiresPrefix14()

assert Valid14.__dict__["field_status"].name == "field_status"
assert events14[-1] == ("Valid14", "field_status")
print("Problem 14 passed: portable diagnosis and a valid binding.")

Problem 14 passed: portable diagnosis and a valid binding.


### Step 3 — Reason about side effects

The first event was appended *even though the class definition failed*. Avoid important external effects in `__set_name__` (opening network connections, registering irrevocable global state, writing files). Prefer local metadata configuration here, and arrange explicit rollback or later registration stages when creation-time failure matters.

---
## Problem 15 — Multiple inheritance: competing fields with the same name

**Scenario.** Two unrelated base classes define descriptor fields named `mode`. A subclass inherits from both. Which descriptor wins on ordinary lookup, and does creating the subclass call `__set_name__` on either inherited descriptor again?

**Challenge:** demonstrate ordinary MRO selection, maintain independent backing storage for the two base descriptors, and use `super()` to deliberately access the next implementation.

### Step 1 — Give each defining owner a separate storage key

Both fields share the *public name* `mode`. If they both store under `instance.__dict__["mode"]`, they will also share the same underlying state even when accessed explicitly. To make the experiment more informative, bind a storage key using **owner name + field name**. (Production code should account for duplicate owner names across modules.)

In [39]:
class CaseField15:
    def __init__(self, transform):
        self.transform = transform
        self.bindings = []

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name
        self.storage_key = f"_{owner.__name__}_{name}"
        self.bindings.append((owner.__name__, name))

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.storage_key not in instance.__dict__:
            raise AttributeError(f"{self.owner.__name__}.{self.name} is unset")
        return instance.__dict__[self.storage_key]

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        instance.__dict__[self.storage_key] = self.transform(value)

class UpperBase15:
    mode = CaseField15(str.upper)

class LowerBase15:
    mode = CaseField15(str.lower)

class Combined15(UpperBase15, LowerBase15):
    pass

print("MRO:", [cls.__name__ for cls in Combined15.__mro__])
print("Declared bindings:", UpperBase15.mode.bindings, LowerBase15.mode.bindings)

MRO: ['Combined15', 'UpperBase15', 'LowerBase15', 'object']
Declared bindings: [('UpperBase15', 'mode')] [('LowerBase15', 'mode')]


### Step 2 — Follow the ordinary lookup route

MRO puts `UpperBase15` before `LowerBase15`, so `Combined15.mode` resolves to the uppercasing descriptor. Ordinary assignment goes through that descriptor as well. The lowercasing descriptor is not configured again for `Combined15`, and its independent backing key is still absent.

In [40]:
combined15 = Combined15()
combined15.mode = "MiXeD"
assert combined15.mode == "MIXED"
assert Combined15.mode is UpperBase15.mode
assert Combined15.mode is not LowerBase15.mode
assert len(UpperBase15.mode.bindings) == 1
assert len(LowerBase15.mode.bindings) == 1
expect_error(AttributeError, lambda: super(UpperBase15, combined15).mode, "LowerBase15")
print("Initial storage:", combined15.__dict__)

Initial storage: {'_UpperBase15_mode': 'MIXED'}


### Step 3 — Access the second implementation intentionally

`super(UpperBase15, combined15).mode` starts searching **after** `UpperBase15`, so it finds `LowerBase15.mode`. We can explicitly invoke that descriptor's setter to give its *separate* backing key a value. Normal lookup still selects the first base's descriptor.

In [41]:
LowerBase15.__dict__["mode"].__set__(combined15, "AnOtHeR")
assert super(UpperBase15, combined15).mode == "another"
assert combined15.mode == "MIXED"
assert combined15.__dict__ == {
    "_UpperBase15_mode": "MIXED",
    "_LowerBase15_mode": "another",
}
print("Problem 15 passed: MRO winner, explicit alternate, no inherited rebinding.")

Problem 15 passed: MRO winner, explicit alternate, no inherited rebinding.


**Takeaway.** `__set_name__` records where a field was declared; MRO determines which inherited attribute is found in ordinary lookup. Deliberate owner-qualified storage can avoid collisions, but an API with two inherited fields sharing a name still needs an explicit conflict policy.

---
# Bonus: four quick diagnostic drills (with answers)

These are short checks to consolidate the concepts. Predict each expression before executing the cell, then compare it with the explanation immediately afterward.

### Drill A — Can a class-level lookup return a descriptor object?

Yes. Returning `self` when `instance is None` is a common convention, not a requirement imposed on every descriptor. Our implementations choose that convention so class access supports introspection and declarative operations.

In [42]:
assert isinstance(User01.identity, IdentityField01)
assert isinstance(Customer06.email, Column06)
assert isinstance(Rectangle11.area, Area11)
print("A: all three class lookups return descriptor objects.")

A: all three class lookups return descriptor objects.


### Drill B — Is a same-name dictionary entry always shadowing?

No. Our data-descriptor area is still invoked with a cached entry under `"area"`; our non-data demonstration in Problem 04 was vulnerable to shared storage because it stored one value *on the descriptor*, not because of dictionary precedence. Keep the two issues separate.

In [43]:
assert rectangle11.__dict__["area"] == 20
assert rectangle11.area == 20
assert Rectangle11.area is Rectangle11.__dict__["area"]
print("B: same-name cache exists; data descriptor still mediates access.")

B: same-name cache exists; data descriptor still mediates access.


### Drill C — Will inheritance always cause a new `__set_name__` call?

No. Inheritance does not rebind the descriptor into the subclass namespace. Our `OwnerProbe09` and `CaseField15` recorded the defining class once per declaration.

In [44]:
assert Base09.__dict__["marker"].binding_events == [("Base09", "marker")]
assert len(UpperBase15.mode.bindings) == 1
print("C: inheriting a descriptor did not create another binding event.")

C: inheriting a descriptor did not create another binding event.


### Drill D — Can a descriptor guard against direct dictionary mutation?

Not by `__set__` alone. The descriptor protocol controls normal attribute operations. It does not intercept a programmer manually changing `instance.__dict__`, and custom attribute machinery can also bypass it. Do not represent it as a security boundary.

In [45]:
probe_d = AuditedWrite08()
probe_d.quantity = 4
probe_d.__dict__["quantity"] = -99  # Deliberately bypasses normal assignment.
assert probe_d.quantity == -99
assert probe_d._writes == [("quantity", 4)]
print("D: direct dictionary mutation bypassed validation and audit logging.")

D: direct dictionary mutation bypassed validation and audit logging.


---
# Consolidated answer key and practical checklist

| Problem | Critical observation | Robust design decision |
|---|---|---|
| 01 | Missing is not `None` | Sentinel or membership; validate before commit |
| 02 | Aliases do not supply name to `__get__` | One descriptor per field, explicit alias |
| 03 | Descriptor reused across different defining classes | Keep per-owner binding metadata; resolve via MRO |
| 04 | Descriptor-level lists are shared | Lazy per-instance factory; know mutation limitations |
| 05 | Declarations exist before instance values | Check partner at binding, compare values on assignment |
| 06 | Class and instance access differ | Build expressions on class, values on instances |
| 07 | Custom `__getattribute__` can bypass descriptors | Delegate to `object.__getattribute__` |
| 08 | Custom `__setattr__` can skip `__set__` | Delegate first, audit only after success |
| 09 | Defining owner differs from access owner | Keep both concepts separate |
| 10 | Auditing requires successful state transitions | Log only after successful writes/deletes |
| 11 | Data descriptors can store same-name cache entries | Explicit dependency invalidation |
| 12 | Descriptors can implement method binding | Use `__get__`, `MethodType`, and callable wrapper |
| 13 | Inherited registries must not share mutable state | Build a fresh schema in `__init_subclass__` |
| 14 | Binding failures can vary by Python version | Inspect original exception/cause; avoid side effects |
| 15 | MRO selects among inherited descriptors | Separate storage and state a conflict policy |

**Design checklist:** Define missing-value behavior; return `self` on class access when useful; state whether descriptor reuse is permitted; avoid accidental shared instance state; validate before committing; explicitly handle inheritance; avoid bypassing normal lookup; test both successful and failing operations.

**Relationship to the supplied notebook:** The source establishes the `__set_name__` hook, uses name-aware validation, demonstrates an accidental backing-name collision, and previews data-descriptor precedence. The cases in this volume are **new extensions** built from those foundations and use the same observation-led teaching style. All exercises are educational examples; several discuss limitations rather than claiming production completeness.